In [6]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from datetime import datetime
import sqlite3
import os


In [7]:
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], 
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

DB_PATH = "attendance.db"


In [8]:
class TokenRequest(BaseModel):
    token: str

class FaceVerificationRequest(BaseModel):
    token: str
    image_data: str
    action_type: str  # "IN" or "OUT"

def get_db_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


In [9]:
@app.get("/")
async def root():
    return {"status": "ok"}

@app.post("/scan-token")
async def scan_token(req: TokenRequest):
    conn = get_db_connection()
    user = conn.execute("SELECT * FROM users WHERE token = ?", (req.token,)).fetchone()
    if not user:
        conn.close()
        raise HTTPException(status_code=404, detail="User not found")
    
    logs = conn.execute("SELECT * FROM daily_logs WHERE token = ? ORDER BY scan_date DESC LIMIT 5", (req.token,)).fetchall()
    conn.close()
    
    logs_data = []
    for log in logs:
        logs_data.append({
            "scan_date": log["scan_date"],
            "in_time": log["in_time"] or "-",
            "out_time": log["out_time"] or "-",
            "in_status": log["in_status"] or "-",
            "out_status": log["out_status"] or "-",
            "duration_status": log["duration_status"] or "-"
        })
    
    return {
        "token": req.token,
        "name": user["name"],
        "next_action": "Please select Clock IN or Clock OUT",
        "logs": logs_data
    }


In [10]:
@app.post("/verify-face")
async def verify_face(req: FaceVerificationRequest):
    conn = get_db_connection()
    user = conn.execute("SELECT * FROM users WHERE token = ?", (req.token,)).fetchone()
    if not user:
        conn.close()
        raise HTTPException(status_code=404, detail="User not found")
        
    now = datetime.now()
    today_date = now.strftime("%Y-%m-%d")
    current_time_str = now.strftime("%H:%M")
    
    log = conn.execute("SELECT * FROM daily_logs WHERE token = ? AND scan_date = ?", (req.token, today_date)).fetchone()
    
    if req.action_type == "IN":
        if log and log["in_time"]:
            conn.close()
            return {"message": "Already clocked in today!", "logs": get_logs_data(req.token), "next_action": "Done"}
            
        in_status = "Late" if now.hour >= 9 else "On Time"
        if not log:
            conn.execute("INSERT INTO daily_logs (token, scan_date, in_time, in_status) VALUES (?, ?, ?, ?)",
                         (req.token, today_date, current_time_str, in_status))
        else:
            conn.execute("UPDATE daily_logs SET in_time = ?, in_status = ? WHERE token = ? AND scan_date = ?",
                         (current_time_str, in_status, req.token, today_date))
        conn.commit()
        msg = f"Clocked IN successfully. Status: {in_status}"
        
    elif req.action_type == "OUT":
        if not log or not log["in_time"]:
            conn.close()
            raise HTTPException(status_code=400, detail="Cannot clock out without clocking in first.")
            
        out_status = "Before Time" if now.hour < 17 else "On Time"
        
        in_time_dt = datetime.strptime(f"{today_date} {log['in_time']}", "%Y-%m-%d %H:%M")
        duration = now - in_time_dt
        duration_hours = duration.total_seconds() / 3600
        
        duration_status = "Minimum time didn't fulfill" if duration_hours < 8 else "Minimum time fulfilled"
        
        conn.execute("""
            UPDATE daily_logs 
            SET out_time = ?, out_status = ?, duration_status = ? 
            WHERE token = ? AND scan_date = ?
        """, (current_time_str, out_status, duration_status, req.token, today_date))
        conn.commit()
        msg = f"Clocked OUT successfully. Status: {out_status}. {duration_status} ({duration_hours:.1f} hrs)"
    else:
        conn.close()
        raise HTTPException(status_code=400, detail="Invalid action type")
    
    logs_data = get_logs_data(req.token)
    conn.close()
        
    return {
        "message": msg,
        "logs": logs_data,
        "next_action": "Done"
    }

def get_logs_data(token):
    conn = get_db_connection()
    logs = conn.execute("SELECT * FROM daily_logs WHERE token = ? ORDER BY scan_date DESC LIMIT 5", (token,)).fetchall()
    conn.close()
    logs_data = []
    for log in logs:
        logs_data.append({
            "scan_date": log["scan_date"],
            "in_time": log["in_time"] or "-",
            "out_time": log["out_time"] or "-",
            "in_status": log["in_status"] or "-",
            "out_status": log["out_status"] or "-",
            "duration_status": log["duration_status"] or "-"
        })
    return logs_data


In [ ]:
import asyncio
nest_asyncio.apply()

# In Jupyter, we are already inside an async event loop,
# so we await the server directly instead of using uvicorn.run()
config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)
await server.serve()


INFO:     Started server process [24216]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:60183 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:54158 - "OPTIONS /scan-token HTTP/1.1" 200 OK
INFO:     127.0.0.1:54158 - "POST /scan-token HTTP/1.1" 200 OK
INFO:     127.0.0.1:57581 - "OPTIONS /verify-face HTTP/1.1" 200 OK
INFO:     127.0.0.1:57581 - "POST /verify-face HTTP/1.1" 200 OK
INFO:     127.0.0.1:54515 - "POST /verify-face HTTP/1.1" 200 OK
